In [1]:
import json
import torch.nn as nn
import torch
from model.Transformer import Transformer,encoder_stack,decoder_stack
from model.Transformer_block import encoder_block,decoder_block
from Data.dataset import Dataset

with open("src_vocab.json", "r") as f:
    src_vocab = json.load(f)

with open("tgt_vocab.json", "r") as f:
    tgt_vocab = json.load(f)

embed_dim = 64

encoder_blocks= [encoder_block(embed_dim,1,128,eps=1e-5) for _ in range(2)]
decoder_blocks= [decoder_block(embed_dim,1,128,eps=1e-5) for _ in range(2)]

enc_stack= encoder_stack(encoder_blocks)
dec_stack= decoder_stack(decoder_blocks)

model= Transformer(enc_stack, dec_stack, embed_dim, len(src_vocab), len(tgt_vocab))

device= torch.device("cuda" if torch.cuda.is_available() else "cpu")

model= model.to(device)

In [6]:
from datasets import load_dataset
import re
from torch.utils.data import DataLoader
from Data.dataloader import collate

def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Zà-ÿ\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

ds = load_dataset("Helsinki-NLP/opus-100", "en-it")

data_pair = []

for sample in ds["train"]:
    en= clean(sample["translation"]["en"])
    it= clean(sample["translation"]["it"])

    data_pair.append((en, it))


dataset= Dataset(data_pair[0:75000],src_vocab_path="src_vocab.json",tgt_vocab_path="tgt_vocab.json", src_vocab=src_vocab,tgt_vocab=tgt_vocab)


train_loader= DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate)



In [ ]:
scaler= torch.amp.GradScaler("cuda")

epochs=10
criterion= torch.nn.CrossEntropyLoss(ignore_index=0)
optimizer= torch.optim.AdamW(model.parameters(), lr=1e-5)

for epoch in range(epochs):

    running_loss=0
    model.train()

    for batch_idx,batch in enumerate(train_loader):

        optimizer.zero_grad()

        src = batch["src"].to(device)
        tgt_in = batch["tgt_in"].to(device)
        tgt_out = batch["tgt_out"].to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):

            output= model(src,tgt_in)
            loss= criterion(output.reshape(-1,len(tgt_vocab)),tgt_out.reshape(-1))


        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print("batch", batch_idx, "loss =", loss.item())

    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f}")






batch 37499 loss = 4.442097187042236
Epoch [1/10] | Loss: 4.9700
batch 37499 loss = 4.078963756561279
Epoch [2/10] | Loss: 4.8604
batch 37499 loss = 5.032718181610107
Epoch [3/10] | Loss: 4.7758
batch 37499 loss = 3.433424472808838
Epoch [4/10] | Loss: 4.6979
batch 37499 loss = 4.717200756072998
Epoch [5/10] | Loss: 4.6361
batch 37499 loss = 4.736558437347412
Epoch [6/10] | Loss: 4.5834
batch 37499 loss = 3.847669839859009
Epoch [7/10] | Loss: 4.5346
batch 37499 loss = 4.59751558303833
Epoch [8/10] | Loss: 4.4887
batch 37499 loss = 4.297616958618164
Epoch [9/10] | Loss: 4.4458
batch 37499 loss = 3.640336036682129
Epoch [10/10] | Loss: 4.4156


# Current status
my optimizer is taking baby steps since the LR is too low and i am using a batch size of 2. this can be improved by gradient accumulation and changing learning rate which i will perform in the coming updates 